In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=(
        "assets/models/qwen3.5-4b/",
        "datasets/flenqa/",
        "checkpoints/flenqa-probe-assets/",
        "runs/flenqa-full-run/model_outputs.parquet",
    ),
)

# FLenQA linear probe evaluation

This notebook validates the frozen probe assets, evaluates them on held-out problems at every context length, and compares answer decodability with the saved model answers. Probes and thresholds are never refit on test data.

In [ ]:
%pip install -qq --disable-pip-version-check pandas matplotlib

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.flenqa_probe_jlens.analysis import validate_split
from experiments.flenqa_probe_jlens.constants import PROBE_CONFIG
from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.evaluation import evaluate_paper_binary
from jlens_reasoning.evaluation_utils import answer_token_variants
from jlens_reasoning.probing import (
    binary_probe_metrics,
    evaluate_probe,
    extract_probe_features,
    load_probe_checkpoint,
    probe_input_contract,
    token_margin,
    validate_probe_input_contract,
)

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
PROBE_PATH = ASSET_DIR / "probes.pt"
METADATA_PATH = ASSET_DIR / "metadata.json"
SPLIT_PATH = context.checkpoints_dir / "flenqa-probe-assets" / "problem_split.json"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
EXPECTED_CONTEXT_SIZES = (250, 500, 1000, 2000, 3000)

## Load and sanity-check the frozen assets

The split is inherited from the training notebook. The saved validation metrics, not test performance, will choose the headline layer.

In [ ]:
checkpoint = load_probe_checkpoint(
    PROBE_PATH,
    metadata_path=METADATA_PATH,
    model_name=MODEL_NAME,
)
metadata = checkpoint["metadata"]
split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
assert metadata["split"] == split_asset
assert metadata["context_sizes"] == [250, 500]
assert "test" not in metadata["example_counts"]
assert all("test" not in metrics for metrics in metadata["probe_metrics"].values())

## Load the model used to fit the probes

Keep the model width and layer count aligned with the frozen checkpoint.

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
input_contract = probe_input_contract(tokenizer, config=PROBE_CONFIG)
validate_probe_input_contract(metadata, input_contract)
num_layers = int(causal_lm.config.num_hidden_layers)
hidden_dim = int(causal_lm.config.hidden_size)
assert metadata["num_layers"] == num_layers
assert metadata["hidden_dim"] == hidden_dim
assert set(checkpoint["layers"]) == set(range(num_layers))

print(
    {
        "layers": num_layers,
        "hidden_dim": hidden_dim,
        "split_sizes": {
            key: len(value) for key, value in split_asset["problems"].items()
        },
        "fit_context_sizes": metadata["context_sizes"],
    }
)

## Evaluate the frozen probes on held-out problems

The probe was fit on 250/500-token prompts only. Applying it to 1000/2000/3000-token prompts is an intentional out-of-distribution test of whether the short-context answer direction remains decodable.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
problem_to_split = validate_split(split_asset, rows)
test_rows = [row for row in rows if problem_to_split[row.problem_id] == "test"]
test_prompts = prepare_prompts(test_rows)
prompt_context_sizes = {}
for prompt in test_prompts:
    context_sizes = {item.ctx_size for item in prompt.provenance}
    assert len(context_sizes) == 1
    prompt_context_sizes[prompt.prompt_id] = context_sizes.pop()
assert set(prompt_context_sizes.values()) == set(EXPECTED_CONTEXT_SIZES)

model_records = pd.read_parquet(MODEL_OUTPUT_PATH).set_index(
    "prompt_id", verify_integrity=True
)
if not {"input_sha256", "n_input_tokens"}.issubset(model_records.columns):
    raise ValueError(
        "Regenerate model answers with the current wheel to record exact chat input hashes"
    )
assert model_records.model_name.eq(MODEL_NAME).all()
assert model_records.inference_mode.eq("direct").all()

test_examples = []
for prompt in test_prompts:
    record = model_records.loc[prompt.prompt_id]
    assert record["problem_id"] == prompt.problem_id
    assert record["label"] == prompt.label and record["text"] == prompt.text
    model_evaluation = evaluate_paper_binary(
        record["generated_text"], expected=prompt.label
    )
    test_examples.append(
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "task": prompt.task,
            "ctx_size": prompt_context_sizes[prompt.prompt_id],
            "label": int(prompt.label),
            "model_answer": model_evaluation.verdict,
            "model_correct": model_evaluation.correct,
            "generation_status": record["generation_status"],
            "inference_mode": record["inference_mode"],
            "input_sha256": record["input_sha256"],
            "n_input_tokens": int(record["n_input_tokens"]),
        }
    )
pd.DataFrame(test_examples).groupby("ctx_size").size()

## Extract chat-prompt hidden states

At each layer, take the final input token from `hidden_states[layer + 1]`. As in training, the last entry is post-final-normalization. Save the raw True-minus-False output margin alongside the probe scores.

In [ ]:
true_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("True",)))
false_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("False",)))
output_margins = []
layer_features = [[] for _ in range(num_layers)]
for prompt in tqdm(test_prompts, desc="Extracting held-out states", unit="prompt"):
    features = extract_probe_features(
        causal_lm,
        tokenizer,
        prompt.text,
        config=PROBE_CONFIG,
        input_record=model_records.loc[prompt.prompt_id],
    )
    output_margins.append(float(token_margin(features.logits, true_ids, false_ids)))
    for layer, state in enumerate(features.states):
        layer_features[layer].append(state)
layer_features = [torch.stack(features) for features in layer_features]

## Score each frozen probe

`(hidden_states - training_mean) @ probe_weight + bias` is positive toward True. Multiply by the gold-label sign for the gold margin; keep the zero threshold fixed. The headline layer is chosen by saved validation log loss.

In [ ]:
# Build the prompt-layer table once; every summary and export uses it below.
test_frame = pd.DataFrame(test_examples)
probe_frames = []
layer_metrics = []
for layer, hidden_states in enumerate(layer_features):
    probe = checkpoint["layers"][layer]
    evaluation = evaluate_probe(probe, hidden_states, test_frame.label.to_numpy())
    probe_frames.append(
        test_frame.assign(
            layer=layer,
            probe_score=evaluation.scores.numpy(),
            gold_margin=evaluation.gold_margins.numpy(),
            gold_probability=evaluation.gold_probabilities.numpy(),
            probe_correct=evaluation.correct.numpy(),
            output_margin=output_margins,
        )
    )
    validation_metrics = metadata["probe_metrics"][str(layer)]["validation"]
    layer_metrics.append(
        {
            "layer": layer,
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_log_loss": validation_metrics["log_loss"],
            "test_accuracy": evaluation.metrics["accuracy"],
            "test_log_loss": evaluation.metrics["log_loss"],
        }
    )

probe_results = pd.concat(probe_frames, ignore_index=True)
layer_table = pd.DataFrame(layer_metrics)
headline_layer = int(
    layer_table.sort_values(["validation_log_loss", "layer"]).iloc[0]["layer"]
)
headline_frame = probe_results[probe_results.layer == headline_layer].copy()
headline_frame["probe_prediction"] = headline_frame.probe_score > 0
display(layer_table)
print(f"Headline layer selected by validation log loss only: {headline_layer}")

## Probe accuracy versus context length

Gold-aligned margins ask whether the correct task label remains linearly decodable from the chat-prompt hidden state, even when the model’s generated answer is wrong.

In [ ]:
context_summary = (
    headline_frame.groupby("ctx_size", as_index=False)
    .agg(
        test_examples=("problem_id", "size"),
        probe_accuracy=("probe_correct", "mean"),
        mean_gold_probability=("gold_probability", "mean"),
        mean_gold_margin=("gold_margin", "mean"),
        model_accuracy=("model_correct", "mean"),
    )
    .sort_values("ctx_size")
)
display(context_summary)
plt.figure(figsize=(7, 4))
plt.plot(
    context_summary["ctx_size"],
    context_summary["model_accuracy"],
    marker="o",
    label="model",
)
plt.plot(
    context_summary["ctx_size"],
    context_summary["probe_accuracy"],
    marker="o",
    label="probe",
)
plt.xticks(EXPECTED_CONTEXT_SIZES)
plt.ylim(0, 1)
plt.xlabel("Nominal context size")
plt.ylabel("Accuracy")
plt.title(f"Held-out accuracy: model vs. probe (layer {headline_layer})")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Model-versus-probe failure cases

A correct probe prediction at long context means the short-context answer direction is still decodable. A wrong probe prediction alone does not show that the information disappeared.

## Raw probe-score distributions

These histograms use the frozen validation-selected layer's raw score, not its sigmoid output. Shared x-axis limits make label separation and shifts in the score location visible as context grows.

In [ ]:
score_bins = np.linspace(
    headline_frame["probe_score"].min(),
    headline_frame["probe_score"].max(),
    25,
)
fig, axes = plt.subplots(
    1, len(EXPECTED_CONTEXT_SIZES), figsize=(14, 3.5), sharex=True, sharey=True
)
for axis, ctx_size in zip(axes, EXPECTED_CONTEXT_SIZES, strict=True):
    group = headline_frame[headline_frame["ctx_size"] == ctx_size]
    for label, color in [(True, "tab:blue"), (False, "tab:orange")]:
        axis.hist(
            group.loc[group["label"] == label, "probe_score"],
            bins=score_bins,
            density=True,
            alpha=0.5,
            color=color,
            label=str(label),
        )
    axis.axvline(0, color="black", linewidth=1)
    axis.set_title(f"context {ctx_size}")
    axis.set_xlabel("raw probe score")
axes[0].set_ylabel("density")
axes[-1].legend(title="gold")
fig.suptitle(f"Headline-layer probe scores (layer {headline_layer})")
plt.tight_layout()
plt.show()

In [ ]:
headline_frame["category"] = np.select(
    [
        headline_frame["model_correct"] & headline_frame["probe_correct"],
        headline_frame["model_correct"] & ~headline_frame["probe_correct"],
        ~headline_frame["model_correct"] & headline_frame["probe_correct"],
    ],
    [
        "model correct / probe correct",
        "model correct / probe wrong",
        "model wrong / probe correct",
    ],
    default="model wrong / probe wrong",
)
failure_counts = (
    headline_frame.groupby(["ctx_size", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=EXPECTED_CONTEXT_SIZES, fill_value=0)
)
failure_percentages = failure_counts.div(failure_counts.sum(axis=1), axis=0).round(3)
display(failure_counts)
display(failure_percentages)

interesting = headline_frame[
    (~headline_frame["model_correct"]) & headline_frame["probe_correct"]
].sort_values(["ctx_size", "gold_probability"], ascending=[False, False])
sample_columns = [
    "problem_id",
    "task",
    "ctx_size",
    "label",
    "model_answer",
    "probe_prediction",
    "gold_probability",
    "gold_margin",
]
display(interesting[sample_columns].head(8))

## Label and task sanity check

These groupings check whether the long-context probe-correct/model-wrong pattern is concentrated in one gold label or task.

In [ ]:
sanity_frame = headline_frame.assign(
    model_wrong=~headline_frame["model_correct"],
    model_wrong_probe_correct=(
        ~headline_frame["model_correct"] & headline_frame["probe_correct"]
    ),
)


def sanity_summary(group_columns):
    summary = sanity_frame.groupby(group_columns, as_index=False, observed=True).agg(
        examples=("problem_id", "size"),
        model_accuracy=("model_correct", "mean"),
        probe_accuracy=("probe_correct", "mean"),
        model_wrong=("model_wrong", "sum"),
        model_wrong_probe_correct=("model_wrong_probe_correct", "sum"),
    )
    summary["p_probe_correct_given_model_wrong"] = (
        summary["model_wrong_probe_correct"] / summary["model_wrong"]
    ).fillna(0.0)
    return summary


label_sanity = sanity_summary(["ctx_size", "label"])
task_label_sanity = sanity_summary(["ctx_size", "task", "label"])
display(label_sanity)
display(task_label_sanity)

## Problem-level failure summary

Prompt variants for one problem are correlated. This summary collapses them to problem flags so the model-wrong / probe-correct pattern is not counted as independent evidence for every variant.

In [ ]:
problem_flags = sanity_frame.groupby(
    ["ctx_size", "problem_id", "label"], as_index=False
).agg(
    model_wrong=("model_wrong", "any"),
    model_wrong_probe_correct=("model_wrong_probe_correct", "any"),
)
problem_failure_summary = sanity_frame.groupby("ctx_size", as_index=False).agg(
    prompt_variants=("prompt_id", "size"),
    unique_problems=("problem_id", "nunique"),
    model_wrong_prompts=("model_wrong", "sum"),
    model_wrong_probe_correct_prompts=("model_wrong_probe_correct", "sum"),
)
problem_counts = problem_flags.groupby("ctx_size", as_index=False).agg(
    unique_model_wrong_problems=("model_wrong", "sum"),
    unique_model_wrong_probe_correct_problems=("model_wrong_probe_correct", "sum"),
)
problem_failure_summary = problem_failure_summary.merge(problem_counts, on="ctx_size")
display(problem_failure_summary)
problem_failure_by_label = problem_flags.groupby(
    ["ctx_size", "label"], as_index=False
).agg(
    unique_model_wrong_problems=("model_wrong", "sum"),
    unique_model_wrong_probe_correct_problems=("model_wrong_probe_correct", "sum"),
)
display(problem_failure_by_label)

## Headline-layer AUROC by context length

AUROC uses the raw probe score, so it tests label separability without relying on the fixed zero threshold.

In [ ]:
all_layer_auroc = pd.DataFrame(
    [
        {
            "layer": layer,
            "ctx_size": ctx_size,
            "auroc": binary_probe_metrics(
                group.label.to_numpy(), torch.tensor(group.probe_score.to_numpy())
            )["auroc"],
        }
        for (layer, ctx_size), group in probe_results.groupby(["layer", "ctx_size"])
    ]
)
headline_auroc = all_layer_auroc.loc[
    all_layer_auroc.layer == headline_layer, ["ctx_size", "auroc"]
]
display(headline_auroc)

## Score drift versus label separation

The overall mean tracks score location, while the True-minus-False difference tracks answer-label separation. Neither quantity changes the frozen probe or its threshold.

In [ ]:
score_summary = (
    headline_frame.assign(
        true_score=headline_frame["probe_score"].where(headline_frame["label"] == 1),
        false_score=headline_frame["probe_score"].where(headline_frame["label"] == 0),
    )
    .groupby("ctx_size", as_index=False)
    .agg(
        mean_true_score=("true_score", "mean"),
        mean_false_score=("false_score", "mean"),
        overall_mean_score=("probe_score", "mean"),
    )
)
score_summary["score_separation"] = (
    score_summary["mean_true_score"] - score_summary["mean_false_score"]
)
score_summary = score_summary.merge(headline_auroc, on="ctx_size")
display(score_summary)
score_summary.set_index("ctx_size")[["overall_mean_score", "score_separation"]].plot(
    marker="o", figsize=(7, 4)
)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Nominal context size")
plt.ylabel("raw probe score")
plt.title("Score location and True/False separation")
plt.grid(alpha=0.25)
plt.show()

## AUROC across layers and context lengths

This descriptive view reuses the already extracted raw scores for every layer. It does not choose a new layer or tune a threshold on the test set.

In [ ]:
all_layer_auroc_pivot = all_layer_auroc.pivot(
    index="layer", columns="ctx_size", values="auroc"
)
display(all_layer_auroc_pivot)
plt.figure(figsize=(8, 4.5))
for ctx_size, group in all_layer_auroc.groupby("ctx_size", sort=True):
    plt.plot(group["layer"], group["auroc"], marker="o", label=str(ctx_size))
plt.axhline(0.5, color="black", linestyle="--", linewidth=1)
plt.xlabel("Layer")
plt.ylabel("AUROC")
plt.title("Raw probe-score AUROC across layers")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend(title="Context size")
plt.show()

## Interpretation limits

Report the computed accuracy, AUROC, and gold-aligned margins; do not assume threshold drift or above-chance AUROC in advance. The probe measures linear decodability of the True/False task label, not retention of all task-relevant information. Its sigmoid score is not a calibrated long-context confidence estimate.

Generated answers, probe states, and next-token margins use the **same direct chat input**. Each prompt is checked against the token-ID/attention-mask hash recorded during generation. The True-minus-False next-token margin is a diagnostic; it is not a graded generated answer. The last probe uses the final normalized state and cannot be passed directly to a pre-normalization J-Lens matrix.

## Short summary

The summary reports descriptive held-out results only; it does not make a causal claim about why model accuracy changes.

In [ ]:
overall_probe_accuracy = float(headline_frame["probe_correct"].mean())
overall_model_accuracy = float(headline_frame["model_correct"].mean())
wrong_model_right_probe = headline_frame[
    (~headline_frame["model_correct"]) & headline_frame["probe_correct"]
]
long_context = headline_frame[headline_frame["ctx_size"] == max(EXPECTED_CONTEXT_SIZES)]
long_interesting = long_context[
    (~long_context["model_correct"]) & long_context["probe_correct"]
]
print(
    {
        "headline_layer": headline_layer,
        "validation_accuracy": float(
            layer_table.loc[
                layer_table["layer"] == headline_layer, "validation_accuracy"
            ].iloc[0]
        ),
        "validation_log_loss": float(
            layer_table.loc[
                layer_table["layer"] == headline_layer, "validation_log_loss"
            ].iloc[0]
        ),
        "overall_test_probe_accuracy": overall_probe_accuracy,
        "overall_test_probe_log_loss": float(
            layer_table.set_index("layer").loc[headline_layer, "test_log_loss"]
        ),
        "overall_test_model_accuracy": overall_model_accuracy,
        "probe_accuracy_by_context": context_summary.set_index("ctx_size")[
            "probe_accuracy"
        ].to_dict(),
        "headline_auroc_by_context": headline_auroc.set_index("ctx_size")[
            "auroc"
        ].to_dict(),
        "model_accuracy_by_context": context_summary.set_index("ctx_size")[
            "model_accuracy"
        ].to_dict(),
        "model_wrong_probe_correct": {
            "overall_count": len(wrong_model_right_probe),
            "overall_fraction": float(
                len(wrong_model_right_probe) / len(headline_frame)
            ),
            "long_context_count": len(long_interesting),
            "long_context_fraction": float(len(long_interesting) / len(long_context)),
        },
    }
)

## Save per-prompt, per-layer results

The primary experiment consumes these small score tables without repeating activation extraction. Hashes bind them to the frozen probes and generated-answer file. Saving overwrites the tables and manifest in `runs/flenqa-probe-eval`; rerun J-Lens analysis afterward.

In [ ]:
result_dir = OUTPUT_DIR / "runs/flenqa-probe-eval"
result_dir.mkdir(parents=True, exist_ok=True)
probe_results.to_parquet(result_dir / "probe_results.parquet", index=False)
all_layer_auroc.to_parquet(result_dir / "auroc.parquet", index=False)
manifest = {
    **input_contract,
    "source_sha256": PROJECT_SOURCE_SHA256,
    "model_name": MODEL_NAME,
    "model_answer_input_format": "chat_template_direct",
    "transformers_version": transformers.__version__,
    "true_ids": list(true_ids),
    "false_ids": list(false_ids),
}
for name, path in (
    ("probes", PROBE_PATH),
    ("model_outputs", MODEL_OUTPUT_PATH),
    ("probe_results", result_dir / "probe_results.parquet"),
    ("auroc", result_dir / "auroc.parquet"),
):
    with path.open("rb") as handle:
        manifest[name + "_sha256"] = hashlib.file_digest(handle, "sha256").hexdigest()
(result_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(f"Saved {len(probe_results):,} prompt-layer scores to {result_dir}")

In [ ]:
upload_artifacts()
print("COLAB_NOTEBOOK_UPLOAD_COMPLETE")